In [ ]:
%pylab inline
import svgpathtools as spt
import eucare as ec
import networkx as nx

In [ ]:
def kawasaki_sum(v):
    angles = np.abs(np.array([e['in_angle'] for e in v.incoming_iter()]))
    assert len(angles) % 2 == 0
    return np.sum(angles * (-1) ** np.arange(len(angles)))
    
def max_kawasaki_sum(vertices):
    if isinstance(vertices, ec.half.HalfEdgeGraph):
        vertices = [v for v in vertices.vertices if not v.on_border()]
    return np.max([kawasaki_sum(v) for v in vertices])

#max_kawasaki_sum(G)

In [ ]:
render_settings = dict(
    width=1500, height=1500,
    figsize=(7, 7),
    scale='auto',
    render_edges=True,
    render_faces=True,
    render_vertices=False,
    line_width=0.003,
    face_inset=0.000,
    for_cutting=False
)

In [ ]:
#filepath = '/home/roman/Documents/Origami/Oripa/tato.svg'
filepath = '/home/roman/Documents/Origami/Oripa/crane_final_mitani.svg'
paths, attributes = spt.svg2paths(filepath)

In [ ]:
#ec.conversions.EHEG_from_edgelist()
#print(attributes)

In [ ]:
points = []
edges = []
for path, attrs in zip(paths, attributes):
    assert len(path) == 1, f'{path}'
    assert isinstance(path[0], spt.path.Line)
    line = path[0]
    start = np.array([line.start.real, line.start.imag], dtype=np.float32)
    end = np.array([line.end.real, line.end.imag], dtype=np.float32)
    #print(start, end)
    crease_type = None
    if 'red' in attrs['style']:
        crease_type = 'mountain'
    elif 'blue' in attrs['style']:
        crease_type = 'valley'
    elif 'gray' in attrs['style']:
        continue
    edge_attrs = dict() if crease_type is None else dict(crease_type=crease_type)
    edges.append((len(points), len(points)+1, edge_attrs))
    points.extend([start, end])
points = np.stack(points)

clustering = ec.overlap.group_closeby(points, 1e-2)

first_occurences = np.argmax(clustering[None] == np.arange(np.max(clustering) + 1)[:, None], axis=1)
merged_points = points[first_occurences]

G = nx.Graph()
G.add_edges_from([(tuple(merged_points[clustering[i]]), tuple(merged_points[clustering[j]]), attrs) 
                  for i, j, attrs in edges])

G = ec.conversions.EHEG_from_nx(G)
G.normalize_positions()

In [ ]:
#cc = ec.classifiers.CountingClassifier(ec.classifiers.lambda_classifier(lambda f: f.area()//0.0001)())
cc = ec.classifiers.congruency_classifier()
for f in G.faces:
    f['color_key'] = cc.classify(f)
G.show(**render_settings)
max_kawasaki_sum(G)

In [ ]:
len(G.faces)

In [ ]:
initial_face = None
for f in G.faces:
    pos = np.array([v['pos'] for v in f.vertex_iter()])
    if np.all(np.min(pos, axis=0) <= 0) and np.all(np.max(pos, axis=0) > 0):
        initial_face = f
G.twocolor_faces(initial_face=initial_face)
G.show(**render_settings)
for f in filter(lambda f: f['color_key'], G.faces):
    for e in f.halfedge_iter():
        e['in_angle'] *= -1
G.recompute_positions()
print(max_kawasaki_sum(G))
G.show(**render_settings)

In [ ]:
def get_over_under_pairs(G, two_coloring_key='color_key'):
    # return list of pairs (f1, f2) with f1 over f2
    # G is assumed to be two-colored
    over_under_pairs = []
    for e in G.halfedges:
        crease_type = e.attributes.get('crease_type', None)
        if crease_type in ('mountain', 'valley') and not (e.on_border() or e.rev.on_border()):
            e_above = e if e.face[two_coloring_key] else e.rev
            if crease_type is 'mountain':
                e_above = e_above.rev
            over_under_pairs.append([e_above.face, e_above.rev.face])
    print('number of pairs', len(over_under_pairs))
    return over_under_pairs

over_under_pairs = get_over_under_pairs(G)
G_over = ec.overlap.overlap_graph(G, eps=1e-5)

In [ ]:
G_over.show(**render_settings)

In [ ]:
from collections import defaultdict
area_threshold = 1e-6

cc = ec.classifiers.CountingClassifier(ec.classifiers.lambda_classifier(lambda f: f.area() > area_threshold)())
counts = defaultdict(int)
for f in G_over.faces:
    # over = 0
    # under = 0
    # for e in f.halfedge_iter():
    #     print(e.attributes)
    #f['color_key'] = over / (over + under)
    f['color_key'] = cc.classify(f)
    counts[f['color_key']] += 1
print(counts)
print()
G_over.show(**render_settings)

In [ ]:
#central_face = next(iter(f for f in G.faces if f.order() == 24))
#over_under_pairs = [(e.rev.face, e.rev.nex.rev.face) for e in central_face.halfedge_iter()]

In [ ]:
ec.overlap.find_face_order(G_over, over_under_pairs, ignore_area_threshold=1e-6)  #, over_under_pairs)

#for e in G.halfedges:
#    print(e['original_face_groups'])


TOP = 'top_side'
BOTTOM = 'bottom_side'


def show_folded(G, side=TOP):
    assert side in (TOP, BOTTOM)
    cc = ec.classifiers.CountingClassifier(ec.classifiers.RepresentationClassifier())
    G = G.copy()
    for f in G.faces:
        #f['color_key'] = len(f['original_faces'])
        try:
            f['color_key'] = cc.classify(f['sorted_original_faces'][0 if side is TOP else -1])
        except IndexError:
            f['color_key'] = 1000
        #print(f['color_key'])
    G.show(**render_settings)

    to_delete = [e
                 for e in G.halfedges
                 if not (e.on_border() or e.rev.on_border()) and e.face['color_key'] is e.rev.face['color_key']]

    G.halfedges.difference_update(to_delete)
    G = ec.conversions.EHEG_from_nx(G.to_networkx_undirected(), {v: v['pos'] for v in G.vertices})
#     to_join = []
#     for v in G.vertices:
#         if not v.on_border() and v.order() == 2:
#             to_join.append(v)
#     for v in to_join:
#         G.join_vertex(v)
    G.recompute_lengths_and_angles()
    cc = ec.classifiers.CountingClassifier(ec.classifiers.lambda_classifier(lambda f: f.area()//0.0001)())
    for f in G.faces:
        # over = 0
        # under = 0
        # for e in f.halfedge_iter():
        #     print(e.attributes)
        #f['color_key'] = over / (over + under)
        f['color_key'] = cc.classify(f)
    G.show(**render_settings)

show_folded(G_over, TOP)
show_folded(G_over, BOTTOM)

In [ ]:
#to solve this properly: every triplet gets an area; 